In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import shutil
import pandas as pd

RAW_ROOT = Path(r"F:\kurs_work\AOD 4\AOD 4")

IMAGES_ROOT = RAW_ROOT / "Images"
LABELS_ROOT = RAW_ROOT / "Annotations" / "YOLOv8 format"

CLEAN_ROOT = Path(r"F:\kurs_work\processed\AOD4_clean")

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

CLASS_NAMES = {
    0: "airplane",
    1: "bird",
    2: "drone",
    3: "helicopter",
}

RECREATE_CLEAN_DATASET = True

print("RAW_ROOT:", RAW_ROOT)
print("CLEAN_ROOT:", CLEAN_ROOT)

RAW_ROOT: F:\kurs_work\AOD 4\AOD 4
CLEAN_ROOT: F:\kurs_work\processed\AOD4_clean


In [4]:
def get_all_images(folder: Path):
    """
    Возвращает изображения без дублей.
    На Windows нельзя искать отдельно *.jpg и *.JPG через glob, потому что можно получить дубли.
    """
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def clamp(value: float, left: float = 0.0, right: float = 1.0) -> float:
    return max(left, min(right, value))


def bbox_from_polygon(coords):
    """
    Преобразует polygon-точки YOLO segmentation-формата в обычный bbox.

    На входе список:
    [x1, y1, x2, y2, x3, y3, ...]
    """
    xs = coords[0::2]
    ys = coords[1::2]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    # На всякий случай обрезаем рамку границами изображения.
    x_min = clamp(x_min)
    x_max = clamp(x_max)
    y_min = clamp(y_min)
    y_max = clamp(y_max)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min

    return x_center, y_center, width, height


def normalize_detection_bbox(x_center, y_center, width, height):
    """
    Проверяет и аккуратно нормализует bbox.

    Если bbox слегка выходит за границы [0; 1], переводим его через x_min/y_min/x_max/y_max
    и обрезаем по границам изображения.
    """
    x_min = x_center - width / 2
    y_min = y_center - height / 2
    x_max = x_center + width / 2
    y_max = y_center + height / 2

    x_min = clamp(x_min)
    y_min = clamp(y_min)
    x_max = clamp(x_max)
    y_max = clamp(y_max)

    new_width = x_max - x_min
    new_height = y_max - y_min
    new_x_center = (x_min + x_max) / 2
    new_y_center = (y_min + y_max) / 2

    return new_x_center, new_y_center, new_width, new_height


def convert_label_to_detection_yolo(label_path: Path):
    """
    Возвращает:
    lines_out: список строк в обычном detection YOLO-формате
    stats: статистика обработки файла
    errors: список ошибок
    """
    lines_out = []
    errors = []

    stats = {
        "bbox_lines": 0,
        "polygon_lines": 0,
        "skipped_lines": 0,
        "empty_label": False,
    }

    if not label_path.exists():
        stats["empty_label"] = True
        return lines_out, stats, errors

    raw_lines = label_path.read_text(encoding="utf-8").splitlines()

    if len(raw_lines) == 0:
        stats["empty_label"] = True
        return lines_out, stats, errors

    for line_number, line in enumerate(raw_lines, start=1):
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        try:
            class_id = int(float(parts[0]))
        except Exception:
            errors.append(f"{label_path}, строка {line_number}: class_id не является числом")
            stats["skipped_lines"] += 1
            continue

        if class_id not in CLASS_NAMES:
            errors.append(f"{label_path}, строка {line_number}: неизвестный class_id={class_id}")
            stats["skipped_lines"] += 1
            continue

        if len(parts) == 5:
            try:
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
            except Exception:
                errors.append(f"{label_path}, строка {line_number}: bbox-координаты не являются числами")
                stats["skipped_lines"] += 1
                continue

            x_center, y_center, width, height = normalize_detection_bbox(
                x_center, y_center, width, height
            )
            stats["bbox_lines"] += 1

        elif len(parts) > 5 and (len(parts) - 1) % 2 == 0:
            try:
                coords = [float(value) for value in parts[1:]]
            except Exception:
                errors.append(f"{label_path}, строка {line_number}: polygon-координаты не являются числами")
                stats["skipped_lines"] += 1
                continue

            if len(coords) < 4:
                errors.append(f"{label_path}, строка {line_number}: слишком мало polygon-точек")
                stats["skipped_lines"] += 1
                continue

            x_center, y_center, width, height = bbox_from_polygon(coords)
            stats["polygon_lines"] += 1

        else:
            errors.append(
                f"{label_path}, строка {line_number}: неизвестный формат, значений: {len(parts)}"
            )
            stats["skipped_lines"] += 1
            continue

        if width <= 0 or height <= 0:
            errors.append(f"{label_path}, строка {line_number}: bbox имеет нулевую ширину или высоту")
            stats["skipped_lines"] += 1
            continue

        lines_out.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    if len(lines_out) == 0:
        stats["empty_label"] = True

    return lines_out, stats, errors

In [5]:
def prepare_clean_dirs():
    if CLEAN_ROOT.exists() and RECREATE_CLEAN_DATASET:
        shutil.rmtree(CLEAN_ROOT)

    for split in SPLITS:
        (CLEAN_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (CLEAN_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    (CLEAN_ROOT / "reports").mkdir(parents=True, exist_ok=True)


def write_data_yaml():
    yaml_text = f"""path: {CLEAN_ROOT.as_posix()}

train: images/train
val: images/valid
test: images/test

names:
"""
    for class_id, class_name in CLASS_NAMES.items():
        yaml_text += f"  {class_id}: {class_name}\n"

    (CLEAN_ROOT / "data.yaml").write_text(yaml_text, encoding="utf-8")


prepare_clean_dirs()

summary_rows = []
label_error_rows = []
object_counter = Counter()
object_counter_by_split = defaultdict(Counter)

for split in SPLITS:
    src_images_dir = IMAGES_ROOT / split
    src_labels_dir = LABELS_ROOT / split / "labels"

    dst_images_dir = CLEAN_ROOT / "images" / split
    dst_labels_dir = CLEAN_ROOT / "labels" / split

    images = get_all_images(src_images_dir)

    print(f"\nОбработка {split}: {len(images)} изображений")

    split_stats = Counter()

    for image_path in images:
        src_label_path = src_labels_dir / f"{image_path.stem}.txt"

        dst_image_path = dst_images_dir / image_path.name
        dst_label_path = dst_labels_dir / f"{image_path.stem}.txt"

        # Копируем изображение без изменения.
        shutil.copy2(image_path, dst_image_path)

        # Конвертируем label к обычному YOLO detection-формату.
        lines_out, label_stats, errors = convert_label_to_detection_yolo(src_label_path)

        dst_label_path.write_text("\n".join(lines_out), encoding="utf-8")

        split_stats["images"] += 1
        split_stats["objects"] += len(lines_out)
        split_stats["empty_labels"] += int(label_stats["empty_label"])
        split_stats["bbox_lines"] += label_stats["bbox_lines"]
        split_stats["polygon_lines"] += label_stats["polygon_lines"]
        split_stats["skipped_lines"] += label_stats["skipped_lines"]

        for line in lines_out:
            class_id = int(line.split()[0])
            object_counter[class_id] += 1
            object_counter_by_split[split][class_id] += 1

        for error in errors:
            label_error_rows.append({
                "split": split,
                "image": image_path.name,
                "label": str(src_label_path),
                "error": error,
            })

    summary_rows.append({
        "split": split,
        "images": split_stats["images"],
        "objects": split_stats["objects"],
        "empty_labels": split_stats["empty_labels"],
        "bbox_lines": split_stats["bbox_lines"],
        "polygon_lines_converted_to_bbox": split_stats["polygon_lines"],
        "skipped_lines": split_stats["skipped_lines"],
    })

write_data_yaml()

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

errors_df = pd.DataFrame(label_error_rows)
display(errors_df.head(20))

summary_df.to_csv(CLEAN_ROOT / "reports" / "clean_dataset_summary.csv", index=False, sep=";", encoding="utf-8-sig")
errors_df.to_csv(CLEAN_ROOT / "reports" / "clean_dataset_errors.csv", index=False, sep=";", encoding="utf-8-sig")

print("\nЧистая копия создана:")
print(CLEAN_ROOT)
print("\ndata.yaml:")
print((CLEAN_ROOT / "data.yaml").read_text(encoding="utf-8"))


Обработка train: 15761 изображений

Обработка valid: 4514 изображений

Обработка test: 2241 изображений


,split,images,objects,empty_labels,bbox_lines,polygon_lines_converted_to_bbox,skipped_lines
0,train,15761,22058,415,22055,3,0
1,valid,4514,6369,125,6369,0,0
2,test,2241,3171,56,3171,0,0


""



Чистая копия создана:
F:\kurs_work\processed\AOD4_clean

data.yaml:
path: F:/kurs_work/processed/AOD4_clean

train: images/train
val: images/valid
test: images/test

names:
  0: airplane
  1: bird
  2: drone
  3: helicopter



In [6]:
# Проверка итоговой структуры и количества файлов

check_rows = []

for split in SPLITS:
    images_count = len(get_all_images(CLEAN_ROOT / "images" / split))
    labels_count = len(list((CLEAN_ROOT / "labels" / split).glob("*.txt")))

    check_rows.append({
        "split": split,
        "images": images_count,
        "labels": labels_count,
        "images_equals_labels": images_count == labels_count,
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

assert all(check_df["images_equals_labels"]), "Количество изображений и label-файлов не совпадает"

if len(errors_df) == 0:
    print("Ошибок при создании чистой копии нет.")
else:
    print("Есть ошибки. Проверь таблицу errors_df и reports/clean_dataset_errors.csv")

,split,images,labels,images_equals_labels
0,train,15761,15761,True
1,valid,4514,4514,True
2,test,2241,2241,True


Ошибок при создании чистой копии нет.
